In [1]:

#importante antes de ejecutar este cuaderno se debe descargar el datasete ptb-xl
#en formato .zip del sitio  https://physionet.org/content/ptb-xl/1.0.3/ ,
# una vez descargado
#subir el archivo .zip a su drive no se debe crear una carpeta para ello solo arrastrar el
#archivo a MiUnidad del drive , de lo contrario tendra problemas con las rutas de los archivos
#una vez ejecutado este cuaderno se creara una carpeta en su drive con el nombre
# agent1-ecg en la cual encontrara los archivos
#   - Dataset completo
#   - TRAIN
#   - VALIDATION
#   - TEST
#en formato hdf5 q sera utlizan para
#entrenar un red neuronal correspondiente al agente ecg.


In [2]:
!pip install wfdb scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.6 MB/s eta 0:00:00


In [3]:
import os
import ast
import numpy as np
import pandas as pd
import wfdb
import h5py

from scipy.signal import resample

In [4]:
# ============================================================
# CONECTAR GOOGLE DRIVE
# Y CREAR CARPETA FINAL DEL AGENTE
# ============================================================

from google.colab import drive
import os

# ------------------------------------------------------------
# MONTAR GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount('/content/drive')

# ------------------------------------------------------------
# CARPETA FINAL
#
# Todos los archivos HDF5 finales generados por este notebook
# se copiarán aquí.
# ------------------------------------------------------------

agent_folder = '/content/drive/MyDrive/agent1-ecg'

os.makedirs(
    agent_folder,
    exist_ok=True
)

print("================================")
print("GOOGLE DRIVE CONECTADO")
print("================================")
print("Carpeta del agente:")
print("✓ Carpeta lista.")


Mounted at /content/drive
GOOGLE DRIVE CONECTADO
Carpeta del agente:
✓ Carpeta lista.


In [5]:
import zipfile

zip_path = (
    '/content/drive/MyDrive/'
    'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip'
)

extract_path = '/content/ptbxl'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset descomprimido correctamente.")

Dataset descomprimido correctamente.


In [6]:
# ============================================================
# RUTAS
# ============================================================

base_path = (
    '/content/ptbxl/'
    'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
)

output_path = (
    '/content/ptbxl_250Hz_LeadII_binary'
)

os.makedirs(
    output_path,
    exist_ok=True
)


print("Dataset:")
print(base_path)

print("\nSalida:")
print(output_path)

Dataset:
/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3

Salida:
/content/ptbxl_250Hz_LeadII_binary


In [7]:
# ============================================================
# COMPROBAR DATASET
# ============================================================

database_file = (
    f'{base_path}/ptbxl_database.csv'
)

scp_file = (
    f'{base_path}/scp_statements.csv'
)


if not os.path.exists(database_file):

    raise FileNotFoundError(
        f"No se encontró:\n{database_file}\n\n"
        "Revisá la variable base_path."
    )


if not os.path.exists(scp_file):

    raise FileNotFoundError(
        f"No se encontró:\n{scp_file}\n\n"
        "Revisá la variable base_path."
    )


print("✓ ptbxl_database.csv encontrado")
print("✓ scp_statements.csv encontrado")

✓ ptbxl_database.csv encontrado
✓ scp_statements.csv encontrado


In [8]:
# ============================================================
# CARGAR BASE DE DATOS
# ============================================================

ptbxl_df = pd.read_csv(
    database_file
)


print(
    "ECG originales:",
    len(ptbxl_df)
)


print("\nColumnas disponibles:")

print(
    ptbxl_df.columns.tolist()
)

ECG originales: 21799

Columnas disponibles:
['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']


In [9]:
# ============================================================
# CARGAR BASE DE DATOS
# ============================================================

ptbxl_df = pd.read_csv(
    database_file
)


print(
    "ECG originales:",
    len(ptbxl_df)
)


print("\nColumnas disponibles:")

print(
    ptbxl_df.columns.tolist()
)

ECG originales: 21799

Columnas disponibles:
['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']


In [10]:
# ============================================================
# FILTRO DE EDAD
# ============================================================

ptbxl_df = ptbxl_df[
    (ptbxl_df['age'] >= 18) &
    (ptbxl_df['age'] < 90)
].copy()


ptbxl_df.reset_index(
    drop=True,
    inplace=True
)


print("================================")
print("FILTRO DE EDAD")
print("================================")


print(
    "ECG restantes:",
    len(ptbxl_df)
)


print(
    "Edad mínima:",
    ptbxl_df['age'].min()
)


print(
    "Edad máxima:",
    ptbxl_df['age'].max()
)

FILTRO DE EDAD
ECG restantes: 21373
Edad mínima: 18.0
Edad máxima: 89.0


In [11]:
# ============================================================
# STRAT FOLD
# ============================================================

print("================================")
print("STRAT FOLD")
print("================================")


print(
    ptbxl_df[
        'strat_fold'
    ]
    .value_counts()
    .sort_index()
)

STRAT FOLD
strat_fold
1     2141
2     2137
3     2146
4     2137
5     2138
6     2124
7     2133
8     2134
9     2132
10    2151
Name: count, dtype: int64


In [12]:
# ============================================================
# CARGAR SCP STATEMENTS
# ============================================================

scp_df = pd.read_csv(
    scp_file,
    index_col=0
)


print(
    "Cantidad de códigos SCP:",
    len(scp_df)
)


print("\nColumnas:")

print(
    scp_df.columns.tolist()
)

Cantidad de códigos SCP: 71

Columnas:
['description', 'diagnostic', 'form', 'rhythm', 'diagnostic_class', 'diagnostic_subclass', 'Statement Category', 'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code', 'DICOM Code']


In [13]:
# ============================================================
# CONVERTIR SCP_CODES
#
# Esta función funciona si scp_codes es:

# 2. string que representa un dict
# ============================================================

def parse_scp_codes(value):

    # --------------------------------------------------------
    # Caso 1:
    # Ya es un diccionario
    # --------------------------------------------------------

    if isinstance(value, dict):

        return value


    # --------------------------------------------------------
    # Caso 2:
    # Es NaN
    # --------------------------------------------------------

    if pd.isna(value):

        return {}


    # --------------------------------------------------------
    # Caso 3:
    # Es texto
    # --------------------------------------------------------

    if isinstance(value, str):

        try:

            parsed = ast.literal_eval(value)

            if isinstance(parsed, dict):

                return parsed

        except Exception:

            pass


    # --------------------------------------------------------
    # Si no se pudo convertir
    # --------------------------------------------------------

    raise ValueError(
        f"No se pudo convertir scp_codes: "
        f"{repr(value)}"
    )


# ============================================================
# APLICAR CONVERSIÓN
# ============================================================

ptbxl_df['scp_codes'] = (
    ptbxl_df['scp_codes']
    .apply(parse_scp_codes)
)


print("✓ scp_codes procesado correctamente.")


# ============================================================
# COMPROBAR
# ============================================================

print("\nTipos:")

print(
    ptbxl_df['scp_codes']
    .head()
    .apply(type)
)

✓ scp_codes procesado correctamente.

Tipos:
0    <class 'dict'>
1    <class 'dict'>
2    <class 'dict'>
3    <class 'dict'>
4    <class 'dict'>
Name: scp_codes, dtype: object


In [14]:
# ============================================================
# CÓDIGOS DIAGNÓSTICOS
# ============================================================

diagnostic_df = scp_df[
    scp_df['diagnostic'] == 1
].copy()


print(
    "Códigos diagnósticos:",
    len(diagnostic_df)
)


Códigos diagnósticos: 44


In [15]:
# ============================================================
# OBTENER DIAGNOSTIC_CLASS
# ============================================================

def get_diagnostic_classes(scp_codes):

    classes = []


    for code in scp_codes.keys():

        if code in diagnostic_df.index:

            diagnostic_class = (
                diagnostic_df.loc[
                    code,
                    'diagnostic_class'
                ]
            )


            if pd.notna(
                diagnostic_class
            ):

                classes.append(
                    diagnostic_class
                )


    return list(
        set(classes)
    )


# ============================================================
# APLICAR
# ============================================================

ptbxl_df[
    'diagnostic_classes'
] = (
    ptbxl_df['scp_codes']
    .apply(get_diagnostic_classes)
)


print(
    ptbxl_df[
        [
            'ecg_id',
            'age',
            'diagnostic_classes',
            'strat_fold'
        ]
    ].head(10)
)

   ecg_id   age diagnostic_classes  strat_fold
0       1  56.0             [NORM]           3
1       2  19.0             [NORM]           2
2       3  37.0             [NORM]           5
3       4  24.0             [NORM]           3
4       5  19.0             [NORM]           4
5       6  18.0             [NORM]           4
6       7  54.0             [NORM]           7
7       8  48.0               [MI]           9
8       9  55.0             [NORM]          10
9      10  22.0             [NORM]           9


In [16]:
# ============================================================
# CLASIFICACIÓN BINARIA
#
# 0 = NORMAL
# 1 = ANORMAL
# ============================================================

def binary_label(classes):

    # --------------------------------------------------------
    # Sin diagnóstico
    # --------------------------------------------------------

    if len(classes) == 0:

        return np.nan


    # --------------------------------------------------------
    # Si existe cualquier diagnóstico diferente de NORM,
    # consideramos el ECG ANORMAL.
    # --------------------------------------------------------

    non_norm_classes = [
        c
        for c in classes
        if c != 'NORM'
    ]


    if len(non_norm_classes) > 0:

        return 1


    # --------------------------------------------------------
    # Únicamente NORM
    # --------------------------------------------------------

    if 'NORM' in classes:

        return 0


    return np.nan


# ============================================================
# CREAR LABEL
# ============================================================

ptbxl_df['label'] = (
    ptbxl_df['diagnostic_classes']
    .apply(binary_label)
)

In [17]:
# ============================================================
# ELIMINAR ECG SIN CLASE DIAGNÓSTICA
# ============================================================

before_labels = len(
    ptbxl_df
)


ptbxl_df = ptbxl_df.dropna(
    subset=['label']
).copy()


ptbxl_df['label'] = (
    ptbxl_df['label']
    .astype(np.int8)
)


ptbxl_df.reset_index(
    drop=True,
    inplace=True
)


after_labels = len(
    ptbxl_df
)


print(
    "ECG antes:",
    before_labels
)


print(
    "ECG después:",
    after_labels
)


print(
    "Eliminados:",
    before_labels - after_labels
)

ECG antes: 21373
ECG después: 20970
Eliminados: 403


In [18]:
# ============================================================
# DISTRIBUCIÓN DE CLASES
# ============================================================

normal_count = (
    ptbxl_df['label'] == 0
).sum()


abnormal_count = (
    ptbxl_df['label'] == 1
).sum()


total = (
    normal_count +
    abnormal_count
)


print("================================")
print("CLASIFICACIÓN BINARIA")
print("================================")


print(
    f"NORMAL (0):   {normal_count}"
)


print(
    f"ANORMAL (1):  {abnormal_count}"
)


print(
    f"TOTAL:        {total}"
)


print("\nPorcentajes:")


print(
    f"NORMAL:  "
    f"{normal_count / total * 100:.2f}%"
)


print(
    f"ANORMAL: "
    f"{abnormal_count / total * 100:.2f}%"
)

CLASIFICACIÓN BINARIA
NORMAL (0):   8941
ANORMAL (1):  12029
TOTAL:        20970

Porcentajes:
NORMAL:  42.64%
ANORMAL: 57.36%


In [19]:
# ============================================================
# FOLD VS CLASE
# ============================================================

fold_class_table = pd.crosstab(
    ptbxl_df['strat_fold'],
    ptbxl_df['label']
)


print(
    fold_class_table
)

label         0     1
strat_fold           
1           889  1217
2           910  1185
3           926  1179
4           872  1221
5           880  1219
6           868  1212
7           904  1188
8           878  1213
9           905  1191
10          909  1204


In [20]:
# ============================================================
# LISTAS
# ============================================================

signals = []

metadata_list = []

errors = []

In [21]:
# ============================================================
# PROCESAMIENTO
#
# 500 Hz
#     ↓
# derivación II
#     ↓
# 250 Hz
#     ↓
# 2500 muestras
#
# SIN NORMALIZACIÓN
# SIN FILTROS ADICIONALES
# ============================================================

for i, row in ptbxl_df.iterrows():

    sample_path = (
        f"{base_path}/"
        f"{row['filename_hr']}"
    )


    try:

        # ====================================================
        # LEER ECG
        # ====================================================

        signal, metadata = wfdb.rdsamp(
            sample_path
        )


        # ====================================================
        # COMPROBAR FRECUENCIA
        # ====================================================

        if metadata['fs'] != 500:

            errors.append({

                'ecg_id':
                    row['ecg_id'],

                'error':
                    (
                        'Frecuencia original '
                        f"incorrecta: "
                        f"{metadata['fs']} Hz"
                    )
            })

            continue


        # ====================================================
        # OBTENER NOMBRES DE DERIVACIONES
        # ====================================================

        lead_names = (
            metadata['sig_name']
        )


        # ====================================================
        # COMPROBAR II
        # ====================================================

        if 'II' not in lead_names:

            errors.append({

                'ecg_id':
                    row['ecg_id'],

                'error':
                    'No contiene derivación II'
            })

            continue


        # ====================================================
        # ÍNDICE DE II
        # ====================================================

        lead_index = (
            lead_names.index('II')
        )


        # ====================================================
        # SELECCIONAR II
        # ====================================================

        signal_II = (
            signal[:, lead_index]
        )


        # ====================================================
        # COMPROBAR LONGITUD ORIGINAL
        # ====================================================

        if len(signal_II) != 5000:

            errors.append({

                'ecg_id':
                    row['ecg_id'],

                'error':
                    (
                        'Longitud original '
                        f"inesperada: "
                        f"{len(signal_II)}"
                    )
            })

            continue


        # ====================================================
        # REMUESTREAR
        #
        # 500 Hz -> 250 Hz
        #
        # 5000 -> 2500
        # ====================================================

        signal_250 = resample(
            signal_II,
            2500
        )


        # ====================================================
        # NO NORMALIZAR
        # NO APLICAR FILTROS
        # ====================================================

        signal_250 = (
            signal_250
            .astype(np.float32)
        )


        # ====================================================
        # AGREGAR SEÑAL
        # ====================================================

        signals.append(
            signal_250
        )


        # ====================================================
        # AGREGAR METADATA
        # ====================================================

        metadata_list.append({

            'ecg_id':
                row['ecg_id'],

            'patient_id':
                row['patient_id'],

            'age':
                row['age'],

            'sex':
                row['sex'],

            'label':
                row['label'],

            'diagnostic_classes':
                str(
                    row['diagnostic_classes']
                ),

            'strat_fold':
                row['strat_fold'],

            'filename_hr':
                row['filename_hr']
        })


    except Exception as e:

        errors.append({

            'ecg_id':
                row['ecg_id'],

            'error':
                str(e)
        })


    # ========================================================
    # PROGRESO
    # ========================================================

    if (i + 1) % 500 == 0:

        print(
            f"Procesados: "
            f"{i + 1} / "
            f"{len(ptbxl_df)}"
        )

Procesados: 500 / 20970
Procesados: 1000 / 20970
Procesados: 1500 / 20970
Procesados: 2000 / 20970
Procesados: 2500 / 20970
Procesados: 3000 / 20970
Procesados: 3500 / 20970
Procesados: 4000 / 20970
Procesados: 4500 / 20970
Procesados: 5000 / 20970
Procesados: 5500 / 20970
Procesados: 6000 / 20970
Procesados: 6500 / 20970
Procesados: 7000 / 20970
Procesados: 7500 / 20970
Procesados: 8000 / 20970
Procesados: 8500 / 20970
Procesados: 9000 / 20970
Procesados: 9500 / 20970
Procesados: 10000 / 20970
Procesados: 10500 / 20970
Procesados: 11000 / 20970
Procesados: 11500 / 20970
Procesados: 12000 / 20970
Procesados: 12500 / 20970
Procesados: 13000 / 20970
Procesados: 13500 / 20970
Procesados: 14000 / 20970
Procesados: 14500 / 20970
Procesados: 15000 / 20970
Procesados: 15500 / 20970
Procesados: 16000 / 20970
Procesados: 16500 / 20970
Procesados: 17000 / 20970
Procesados: 17500 / 20970
Procesados: 18000 / 20970
Procesados: 18500 / 20970
Procesados: 19000 / 20970
Procesados: 19500 / 20970
Proces

In [22]:
# ============================================================
# CONVERTIR SEÑALES
# ============================================================

X = np.asarray(
    signals,
    dtype=np.float32
)


# ============================================================
# CONVERTIR METADATA
# ============================================================

metadata_final = pd.DataFrame(
    metadata_list
)


# ============================================================
# TIPOS
# ============================================================

metadata_final['label'] = (
    metadata_final['label']
    .astype(np.int8)
)


metadata_final['strat_fold'] = (
    metadata_final['strat_fold']
    .astype(np.int8)
)


print(
    "Shape X:",
    X.shape
)


print(
    "Shape metadata:",
    metadata_final.shape
)

Shape X: (20970, 2500)
Shape metadata: (20970, 8)


In [23]:
# ============================================================
# COMPROBACIONES
# ============================================================

assert len(X) == len(
    metadata_final
)


assert X.shape[1] == 2500


assert (
    metadata_final['label']
    .isin([0, 1])
    .all()
)


assert (
    metadata_final['strat_fold']
    .isin(range(1, 11))
    .all()
)


print("================================")
print("COMPROBACIONES")
print("================================")


print(
    "Cantidad ECG:",
    X.shape[0]
)


print(
    "Muestras ECG:",
    X.shape[1]
)


print(
    "Derivación: II"
)


print(
    "Frecuencia: 250 Hz"
)


print(
    "Duración:",
    2500 / 250,
    "segundos"
)


print(
    "Normalización: NO"
)


print(
    "Filtro adicional: NO"
)


print(
    "✓ Todas las comprobaciones correctas."
)

COMPROBACIONES
Cantidad ECG: 20970
Muestras ECG: 2500
Derivación: II
Frecuencia: 250 Hz
Duración: 10.0 segundos
Normalización: NO
Filtro adicional: NO
✓ Todas las comprobaciones correctas.


In [24]:
# ============================================================
# CALIDAD DE LAS SEÑALES
# ============================================================

nan_count = np.isnan(X).sum()

inf_count = np.isinf(X).sum()


print(
    "NaN:",
    nan_count
)


print(
    "Infinitos:",
    inf_count
)


assert nan_count == 0
assert inf_count == 0


print(
    "\n✓ No hay NaN ni infinitos."
)

NaN: 0
Infinitos: 0

✓ No hay NaN ni infinitos.


In [25]:
# ============================================================
# GUARDAR SEÑALES
# ============================================================

npy_path = (
    f'{output_path}/'
    'X_250Hz_LeadII.npy'
)


np.save(
    npy_path,
    X
)


print(
    "Guardado:",
    npy_path
)

Guardado: /content/ptbxl_250Hz_LeadII_binary/X_250Hz_LeadII.npy


In [26]:
# ============================================================
# GUARDAR METADATA
# ============================================================

csv_path = (
    f'{output_path}/'
    'metadata.csv'
)


metadata_final.to_csv(
    csv_path,
    index=False
)


print(
    "Guardado:",
    csv_path
)

Guardado: /content/ptbxl_250Hz_LeadII_binary/metadata.csv


In [27]:
# ============================================================
# GUARDAR DATASET NPZ
# ============================================================

npz_path = (
    f'{output_path}/'
    'PTBXL_250Hz_LeadII_binary.npz'
)


np.savez_compressed(

    npz_path,

    X=X,

    y=metadata_final[
        'label'
    ].values,

    ecg_id=metadata_final[
        'ecg_id'
    ].values,

    patient_id=metadata_final[
        'patient_id'
    ].values,

    age=metadata_final[
        'age'
    ].values,

    sex=metadata_final[
        'sex'
    ].values,

    strat_fold=metadata_final[
        'strat_fold'
    ].values
)


print(
    "Guardado:",
    npz_path
)

Guardado: /content/ptbxl_250Hz_LeadII_binary/PTBXL_250Hz_LeadII_binary.npz


In [28]:
# ============================================================
# GUARDAR ERRORES DE PROCESAMIENTO
# ============================================================

errors_df = pd.DataFrame(
    errors
)


errors_path = (
    f'{output_path}/'
    'processing_errors.csv'
)


errors_df.to_csv(
    errors_path,
    index=False
)


print(
    "ECG con errores:",
    len(errors_df)
)


print(
    "Archivo:",
    errors_path
)


if len(errors_df) > 0:

    display(
        errors_df.head(20)
    )

ECG con errores: 0
Archivo: /content/ptbxl_250Hz_LeadII_binary/processing_errors.csv


In [29]:
# ============================================================
# CREAR HDF5 DEFINITIVO
# ============================================================

h5_path = (
    f'{output_path}/'
    'PTB_XL_senal_250Hz_derivacion_II.h5'
)


with h5py.File(
    h5_path,
    'w'
) as f:

    # ========================================================
    # SEÑALES
    # ========================================================

    f.create_dataset(
        'X',
        data=X,
        compression='gzip',
        compression_opts=4
    )


    # ========================================================
    # ETIQUETAS
    #
    # 0 = NORMAL
    # 1 = ANORMAL
    # ========================================================

    f.create_dataset(
        'y',
        data=metadata_final['label'].values,
        compression='gzip'
    )


    # ========================================================
    # IDENTIFICADORES
    # ========================================================

    f.create_dataset(
        'ecg_id',
        data=metadata_final['ecg_id'].values
    )

    f.create_dataset(
        'patient_id',
        data=metadata_final['patient_id'].values
    )


    # ========================================================
    # METADATOS
    # ========================================================

    f.create_dataset(
        'age',
        data=metadata_final['age'].values
    )

    f.create_dataset(
        'sex',
        data=metadata_final['sex'].values
    )


    # ========================================================
    # STRAT FOLD
    # ========================================================

    f.create_dataset(
        'strat_fold',
        data=metadata_final['strat_fold'].values
    )


    # ========================================================
    # ATRIBUTOS
    # ========================================================

    f.attrs['dataset'] = 'PTB-XL'

    f.attrs['sampling_frequency'] = 250

    f.attrs['original_sampling_frequency'] = 500

    f.attrs['lead'] = 'II'

    f.attrs['duration_seconds'] = 10

    f.attrs['samples_per_ecg'] = 2500

    f.attrs['age_min'] = 18

    f.attrs['age_max'] = 88

    f.attrs['normalization'] = 'None'

    f.attrs['filtering'] = 'None'

    f.attrs['label_0'] = 'NORMAL'

    f.attrs['label_1'] = 'ANORMAL'


print("================================")
print("HDF5 CREADO CORRECTAMENTE")
print("================================")

print(
    h5_path
)

HDF5 CREADO CORRECTAMENTE
/content/ptbxl_250Hz_LeadII_binary/PTB_XL_senal_250Hz_derivacion_II.h5


In [30]:
# ============================================================
# INSPECCIONAR HDF5
# ============================================================

with h5py.File(
    h5_path,
    'r'
) as f:


    print(
        "================================"
    )

    print(
        "CONTENIDO HDF5"
    )

    print(
        "================================\n"
    )


    for key in f.keys():

        print(
            f'{key}: '
            f'shape={f[key].shape}, '
            f'dtype={f[key].dtype}'
        )


    print(
        "\n================================"
    )

    print(
        "ATRIBUTOS"
    )

    print(
        "================================"
    )


    for key, value in f.attrs.items():

        print(
            f'{key}: {value}'
        )

CONTENIDO HDF5

X: shape=(20970, 2500), dtype=float32
age: shape=(20970,), dtype=float64
ecg_id: shape=(20970,), dtype=int64
patient_id: shape=(20970,), dtype=float64
sex: shape=(20970,), dtype=int64
strat_fold: shape=(20970,), dtype=int8
y: shape=(20970,), dtype=int8

ATRIBUTOS
age_max: 88
age_min: 18
dataset: PTB-XL
duration_seconds: 10
filtering: None
label_0: NORMAL
label_1: ANORMAL
lead: II
normalization: None
original_sampling_frequency: 500
samples_per_ecg: 2500
sampling_frequency: 250


In [31]:
# ============================================================
# DISTRIBUCIÓN FINAL
# ============================================================

with h5py.File(
    h5_path,
    'r'
) as f:

    y = f['y'][:]

    folds = f['strat_fold'][:]


print(
    "================================"
)

print(
    "DISTRIBUCIÓN FINAL"
)

print(
    "================================"
)


for fold in sorted(
    np.unique(folds)
):

    fold_mask = (
        folds == fold
    )


    total_fold = np.sum(
        fold_mask
    )


    normal_fold = np.sum(
        fold_mask &
        (y == 0)
    )


    abnormal_fold = np.sum(
        fold_mask &
        (y == 1)
    )


    print(
        f'Fold {fold}: '
        f'{total_fold} ECG | '
        f'NORMAL={normal_fold} | '
        f'ANORMAL={abnormal_fold}'
    )

DISTRIBUCIÓN FINAL
Fold 1: 2106 ECG | NORMAL=889 | ANORMAL=1217
Fold 2: 2095 ECG | NORMAL=910 | ANORMAL=1185
Fold 3: 2105 ECG | NORMAL=926 | ANORMAL=1179
Fold 4: 2093 ECG | NORMAL=872 | ANORMAL=1221
Fold 5: 2099 ECG | NORMAL=880 | ANORMAL=1219
Fold 6: 2080 ECG | NORMAL=868 | ANORMAL=1212
Fold 7: 2092 ECG | NORMAL=904 | ANORMAL=1188
Fold 8: 2091 ECG | NORMAL=878 | ANORMAL=1213
Fold 9: 2096 ECG | NORMAL=905 | ANORMAL=1191
Fold 10: 2113 ECG | NORMAL=909 | ANORMAL=1204


In [32]:
# ============================================================
# CREAR PARTICIONES TRAIN / VALIDATION / TEST
#
# ARCHIVO ORIGINAL:
#
# PTB_XL_senal_250Hz_derivacion_II.h5
#
# ARCHIVOS GENERADOS:
#
# TRAIN_PTB_XL_senal_250Hz_derivacion_II.h5
# VAL_PTB_XL_senal_250Hz_derivacion_II.h5
# TEST_PTB_XL_senal_250Hz_derivacion_II.h5
#
# Las particiones se realizan utilizando strat_fold.
# ============================================================


# ============================================================
# LEER STRAT_FOLD DEL HDF5 ORIGINAL
# ============================================================

with h5py.File(
    h5_path,
    'r'
) as f:

    folds = f['strat_fold'][:]


# ============================================================
# DEFINIR PARTICIONES
#
# Fold 1-8  -> TRAIN
# Fold 9    -> VALIDATION
# Fold 10   -> TEST
# ============================================================

train_idx = np.isin(
    folds,
    [1, 2, 3, 4, 5, 6, 7, 8]
)

val_idx = (
    folds == 9
)

test_idx = (
    folds == 10
)


# ============================================================
# COMPROBAR PARTICIONES
# ============================================================

print("================================")
print("PARTICIONES")
print("================================")

print(
    "TRAIN:",
    np.sum(train_idx)
)

print(
    "VALIDATION:",
    np.sum(val_idx)
)

print(
    "TEST:",
    np.sum(test_idx)
)

print(
    "TOTAL:",
    len(folds)
)


assert (
    np.sum(train_idx)
    +
    np.sum(val_idx)
    +
    np.sum(test_idx)
    ==
    len(folds)
)


# ============================================================
# DEFINIR NOMBRES
# ============================================================

train_path = (
    f'{output_path}/'
    'TRAIN_PTB_XL_senal_250Hz_derivacion_II.h5'
)

val_path = (
    f'{output_path}/'
    'VAL_PTB_XL_senal_250Hz_derivacion_II.h5'
)

test_path = (
    f'{output_path}/'
    'TEST_PTB_XL_senal_250Hz_derivacion_II.h5'
)


# ============================================================
# FUNCIÓN PARA CREAR CADA PARTICIÓN
# ============================================================

def create_partition(
    source_path,
    destination_path,
    indices,
    partition_name
):

    # --------------------------------------------------------
    # Abrir HDF5 original
    # --------------------------------------------------------

    with h5py.File(
        source_path,
        'r'
    ) as source:

        # ----------------------------------------------------
        # Crear nuevo HDF5
        # ----------------------------------------------------

        with h5py.File(
            destination_path,
            'w'
        ) as destination:

            # =================================================
            # COPIAR DATASETS
            # =================================================

            dataset_names = [
                'X',
                'y',
                'ecg_id',
                'patient_id',
                'age',
                'sex',
                'strat_fold'
            ]


            for name in dataset_names:

                data = source[name][indices]

                destination.create_dataset(

                    name,

                    data=data,

                    compression='gzip',

                    compression_opts=4
                )


            # =================================================
            # COPIAR ATRIBUTOS DEL HDF5 ORIGINAL
            # =================================================

            for key, value in source.attrs.items():

                destination.attrs[
                    key
                ] = value


            # =================================================
            # AGREGAR INFORMACIÓN DE PARTICIÓN
            # =================================================

            destination.attrs[
                'partition'
            ] = partition_name


            destination.attrs[
                'source_file'
            ] = os.path.basename(
                source_path
            )


    print(
        f"✓ {partition_name} creado:"
    )

    print(
        f"  {destination_path}"
    )


# ============================================================
# CREAR TRAIN
# ============================================================

create_partition(

    h5_path,

    train_path,

    train_idx,

    'TRAIN'
)


# ============================================================
# CREAR VALIDATION
# ============================================================

create_partition(

    h5_path,

    val_path,

    val_idx,

    'VALIDATION'
)


# ============================================================
# CREAR TEST
# ============================================================

create_partition(

    h5_path,

    test_path,

    test_idx,

    'TEST'
)


print()
print("================================")
print("PARTICIONES CREADAS")
print("================================")

print(
    "TRAIN:",
    train_path
)

print(
    "VALIDATION:",
    val_path
)

print(
    "TEST:",
    test_path
)

PARTICIONES
TRAIN: 16761
VALIDATION: 2096
TEST: 2113
TOTAL: 20970
✓ TRAIN creado:
  /content/ptbxl_250Hz_LeadII_binary/TRAIN_PTB_XL_senal_250Hz_derivacion_II.h5
✓ VALIDATION creado:
  /content/ptbxl_250Hz_LeadII_binary/VAL_PTB_XL_senal_250Hz_derivacion_II.h5
✓ TEST creado:
  /content/ptbxl_250Hz_LeadII_binary/TEST_PTB_XL_senal_250Hz_derivacion_II.h5

PARTICIONES CREADAS
TRAIN: /content/ptbxl_250Hz_LeadII_binary/TRAIN_PTB_XL_senal_250Hz_derivacion_II.h5
VALIDATION: /content/ptbxl_250Hz_LeadII_binary/VAL_PTB_XL_senal_250Hz_derivacion_II.h5
TEST: /content/ptbxl_250Hz_LeadII_binary/TEST_PTB_XL_senal_250Hz_derivacion_II.h5


In [33]:
# ============================================================
# GUARDAR LOS HDF5 FINALES EN GOOGLE DRIVE
#
# CARPETA:
# /content/drive/MyDrive/agent1-ecg
#
# SOLO SE COPIAN LOS ARCHIVOS .h5 FINALES GENERADOS
# POR ESTE NOTEBOOK.
# ============================================================

import os
import shutil

# ------------------------------------------------------------
# COMPROBAR QUE GOOGLE DRIVE ESTÁ MONTADO
# ------------------------------------------------------------

if not os.path.isdir('/content/drive/MyDrive'):
    raise RuntimeError(
        "Google Drive no está montado. "
        "Ejecutá primero la celda de conexión a Drive."
    )

# ------------------------------------------------------------
# CARPETA DE DESTINO
# ------------------------------------------------------------

agent_folder = (
    '/content/drive/MyDrive/'
    'agent1-ecg'
)

os.makedirs(
    agent_folder,
    exist_ok=True
)

# ------------------------------------------------------------
# CARPETA LOCAL DONDE SE GENERARON LOS HDF5
# ------------------------------------------------------------

source_folder = (
    '/content/ptbxl_250Hz_LeadII_binary'
)

# ------------------------------------------------------------
# ARCHIVOS HDF5 FINALES
#
# 1. Dataset completo
# 2. Train
# 3. Validation
# 4. Test
# ------------------------------------------------------------

files_to_copy = [
    'PTB_XL_senal_250Hz_derivacion_II.h5',
    'TRAIN_PTB_XL_senal_250Hz_derivacion_II.h5',
    'VAL_PTB_XL_senal_250Hz_derivacion_II.h5',
    'TEST_PTB_XL_senal_250Hz_derivacion_II.h5'
]

# ------------------------------------------------------------
# COMPROBAR QUE TODOS LOS HDF5 FUERON CREADOS
# ANTES DE COPIARLOS
# ------------------------------------------------------------

missing_files = []

for filename in files_to_copy:

    source = os.path.join(
        source_folder,
        filename
    )

    if not os.path.isfile(source):
        missing_files.append(filename)

if missing_files:

    raise FileNotFoundError(
        "No se encontraron los siguientes HDF5 finales:\n"
        + "\n".join(
            f"- {filename}"
            for filename in missing_files
        )
        + "\n\nEl notebook se detiene para evitar "
          "copiar un resultado incompleto a Google Drive."
    )

# ------------------------------------------------------------
# COPIAR / SOBRESCRIBIR LOS HDF5
# ------------------------------------------------------------

print("================================")
print("COPIANDO HDF5 FINALES")
print("================================")

for filename in files_to_copy:

    source = os.path.join(
        source_folder,
        filename
    )

    destination = os.path.join(
        agent_folder,
        filename
    )

    shutil.copy2(
        source,
        destination
    )

    size_mb = (
        os.path.getsize(destination)
        / (1024 ** 2)
    )

    print(
        f"✓ {filename} "
        f"({size_mb:.2f} MB)"
    )

# ------------------------------------------------------------
# VERIFICACIÓN FINAL
# ------------------------------------------------------------

print()
print("================================")
print("VERIFICACIÓN GOOGLE DRIVE")
print("================================")

all_ok = True

for filename in files_to_copy:

    filepath = os.path.join(
        agent_folder,
        filename
    )

    if os.path.isfile(filepath):

        size_mb = (
            os.path.getsize(filepath)
            / (1024 ** 2)
        )

        print(
            f"✓ {filename} "
            f"({size_mb:.2f} MB)"
        )

    else:

        all_ok = False

        print(
            f"✗ FALTA: {filename}"
        )

if not all_ok:

    raise RuntimeError(
        "La verificación final de Google Drive "
        "detectó archivos faltantes."
    )

print()
print("================================")
print("PROCESO FINALIZADO CORRECTAMENTE")
print("================================")
print()
print("Los HDF5 finales están en:")
print(agent_folder)
print()
print("Archivos guardados:")

for filename in files_to_copy:
    print(f"  - {filename}")


COPIANDO HDF5 FINALES
✓ PTB_XL_senal_250Hz_derivacion_II.h5 (186.76 MB)
✓ TRAIN_PTB_XL_senal_250Hz_derivacion_II.h5 (148.88 MB)
✓ VAL_PTB_XL_senal_250Hz_derivacion_II.h5 (18.65 MB)
✓ TEST_PTB_XL_senal_250Hz_derivacion_II.h5 (18.80 MB)

VERIFICACIÓN GOOGLE DRIVE
✓ PTB_XL_senal_250Hz_derivacion_II.h5 (186.76 MB)
✓ TRAIN_PTB_XL_senal_250Hz_derivacion_II.h5 (148.88 MB)
✓ VAL_PTB_XL_senal_250Hz_derivacion_II.h5 (18.65 MB)
✓ TEST_PTB_XL_senal_250Hz_derivacion_II.h5 (18.80 MB)

PROCESO FINALIZADO CORRECTAMENTE

Los HDF5 finales están en:
/content/drive/MyDrive/agent1-ecg

Archivos guardados:
  - PTB_XL_senal_250Hz_derivacion_II.h5
  - TRAIN_PTB_XL_senal_250Hz_derivacion_II.h5
  - VAL_PTB_XL_senal_250Hz_derivacion_II.h5
  - TEST_PTB_XL_senal_250Hz_derivacion_II.h5


In [34]:
#La celdas siguiente no es necesario fue para verficarar algo no mas

In [35]:
import numpy as np
import pandas as pd

# Suponiendo que 'metadata_final' tiene las columnas 'patient_id' y 'strat_fold'
train_folds = [1, 2, 3, 4, 5, 6, 7, 8]

patients_train = set(metadata_final[metadata_final['strat_fold'].isin(train_folds)]['patient_id'])
patients_val   = set(metadata_final[metadata_final['strat_fold'] == 9]['patient_id'])
patients_test  = set(metadata_final[metadata_final['strat_fold'] == 10]['patient_id'])

# Comprobar intersecciones (pacientes repetidos entre particiones)
overlap_train_val = patients_train.intersection(patients_val)
overlap_train_test = patients_train.intersection(patients_test)
overlap_val_test = patients_val.intersection(patients_test)

print(f"Pacientes repetidos entre Train y Val:   {len(overlap_train_val)}")
print(f"Pacientes repetidos entre Train y Test:  {len(overlap_train_test)}")
print(f"Pacientes repetidos entre Val y Test:    {len(overlap_val_test)}")

if len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0:
    print("\n¡Verificación exitosa! No hay cruce de pacientes entre las particiones.")
else:
    print("\¡Alerta! Hay superposición de pacientes.")

Pacientes repetidos entre Train y Val:   0
Pacientes repetidos entre Train y Test:  0
Pacientes repetidos entre Val y Test:    0

¡Verificación exitosa! No hay cruce de pacientes entre las particiones.
